In [ ]:
# Cell 1 — Imports  (pyarrow MUST be first: Windows DLL fix)
import pyarrow
import transformers
import torch
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tqdm import tqdm

print(f'torch {torch.__version__} | CUDA: {torch.cuda.is_available()}')

In [ ]:
# Cell 2 — Config
MODEL_NAME     = 'gpt2'
DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'
N_SAMPLES      = 100
NLDD_THRESHOLD = 0.1
OUT_DIR        = '../outputs'
PROBE_PATH     = os.path.join(OUT_DIR, 'probe_best.pkl')

os.makedirs(OUT_DIR, exist_ok=True)
print(f'Model: {MODEL_NAME}  |  Device: {DEVICE}')
print(f'Probe will be saved to: {PROBE_PATH}')

In [ ]:
# Cell 3 — Load GPT-2 + 20 GSM8K samples
from src.utils.model_loader import get_model
from data.dataset_loader import load_gsm8k

print(f'Loading {MODEL_NAME} ...')
model = get_model(MODEL_NAME, device=DEVICE)
print(f'Loaded  layers={model.cfg.n_layers}  d_model={model.cfg.d_model}')

samples = load_gsm8k(split='test', max_samples=N_SAMPLES)
print(f'Loaded {len(samples)} GSM8K samples')

In [ ]:
# Cell 4 — Generate faithfulness labels and collect last-token activations
from src.probing.faithfulness_detector import generate_faithfulness_labels
from src.probing.cot_probe import extract_cot_activations

# all_acts_by_layer[l] = list of 1-D activation vectors (one per sample)
all_acts_by_layer = {l: [] for l in range(model.cfg.n_layers)}
all_labels        = []   # True = any step faithful
per_step_acts     = {l: [] for l in range(model.cfg.n_layers)}
per_step_labels   = []

for sample in tqdm(samples, desc='Extracting activations'):
    question, steps = sample['question'], sample['steps']
    if len(steps) < 2:
        continue
    prompt = 'Q: ' + question + ' A: ' + ' '.join(steps)
    try:
        label_tuples = generate_faithfulness_labels(
            model, prompt, steps,
            strategy='number_perturbation', nldd_threshold=NLDD_THRESHOLD)
    except Exception:
        continue
    acts = extract_cot_activations(model, prompt)
    # Sample-level: label = majority faithful across steps
    n_faithful = sum(1 for _, lbl, _ in label_tuples if lbl)
    sample_label = n_faithful > len(label_tuples) / 2
    for l in range(model.cfg.n_layers):
        all_acts_by_layer[l].append(acts[l].mean(dim=0).numpy())
    all_labels.append(sample_label)
    # Per-step (for richer probe training)
    for step_idx, is_faithful, _ in label_tuples:
        for l in range(model.cfg.n_layers):
            per_step_acts[l].append(acts[l].mean(dim=0).numpy())
        per_step_labels.append(is_faithful)

print(f'Samples collected: {len(all_labels)}')
print(f'Per-step samples: {len(per_step_labels)}')
print(f'Faithful fraction: {sum(per_step_labels)/max(1,len(per_step_labels)):.1%}')

In [ ]:
# Cell 5 — Train probe per layer, print CV accuracy, Figure 2a bar chart
from src.probing.faithfulness_detector import FaithfulnessProbe
from src.utils.visualizer import plot_probe_accuracy_by_layer

layer_accuracies = {}
probes           = {}

train_acts   = per_step_acts   if len(per_step_labels) >= 4 else all_acts_by_layer
train_labels = per_step_labels if len(per_step_labels) >= 4 else all_labels

print(f'Training probe at each of {model.cfg.n_layers} layers...')
for l in tqdm(range(model.cfg.n_layers), desc='Layer probes'):
    acts_arr = np.array(train_acts[l])
    if len(acts_arr) < 4 or len(set(train_labels)) < 2:
        continue
    probe = FaithfulnessProbe(layer=l)
    cv_acc = probe.train(acts_arr, train_labels)
    layer_accuracies[l] = cv_acc
    probes[l] = probe

if layer_accuracies:
    best_layer = max(layer_accuracies, key=layer_accuracies.get)
    print(f'Best layer: {best_layer}  CV accuracy: {layer_accuracies[best_layer]:.3f}')
    for l, acc in sorted(layer_accuracies.items()):
        print(f'  Layer {l:2d}: {acc:.3f}')
else:
    best_layer = model.cfg.n_layers // 2
    print('Insufficient data for per-layer comparison; using middle layer.')

# Figure 2a — probe accuracy by layer
if layer_accuracies:
    fig = plot_probe_accuracy_by_layer(layer_accuracies)
    fig.savefig(os.path.join(OUT_DIR, 'figure2a_probe_accuracy.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved -> outputs/figure2a_probe_accuracy.png')

In [ ]:
# Cell 6 — Confusion matrix + confidence distribution (Figure 2b)
best_probe = probes.get(best_layer)
if best_probe is None:
    print('No probe trained; skipping Figure 2b.')
else:
    acts_arr = np.array(train_acts[best_layer])
    preds    = best_probe.predict(acts_arr)
    pred_labels = [lbl for lbl, _ in preds]
    confidences = [conf for _, conf in preds]

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    # Confusion matrix
    cm = confusion_matrix(train_labels, pred_labels)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Unfaithful', 'Faithful'])
    disp.plot(ax=axes[0], colorbar=False)
    axes[0].set_title(f'Confusion Matrix — Layer {best_layer}')

    # Confidence distribution
    faith_conf = [c for c, l in zip(confidences, train_labels) if l]
    unfai_conf = [c for c, l in zip(confidences, train_labels) if not l]
    axes[1].hist(faith_conf, bins=10, alpha=0.65, color='steelblue', label='Faithful')
    axes[1].hist(unfai_conf, bins=10, alpha=0.65, color='salmon',    label='Unfaithful')
    axes[1].axvline(0.5, color='black', ls='--', lw=1, label='threshold=0.5')
    axes[1].set(xlabel='Probe confidence', ylabel='Count',
                title=f'Confidence Distribution — Layer {best_layer}')
    axes[1].legend(fontsize=8)

    fig.suptitle('Figure 2b: Probe Evaluation', fontsize=12)
    plt.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, 'figure2b_probe_eval.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved -> outputs/figure2b_probe_eval.png')

In [ ]:
# Cell 7 — PCA scatter + probe direction arrow (Figure 2c)
from src.utils.visualizer import plot_pca_with_direction

if best_probe is not None:
    acts_arr  = np.array(train_acts[best_layer])
    direction = best_probe.direction  # shape (d_model,)
    fig = plot_pca_with_direction(acts_arr, direction)
    fig.set_size_inches(6, 6)
    axes_list = fig.get_axes()
    axes_list[0].set_title(f'Figure 2c: PCA + Probe Direction (Layer {best_layer})')
    fig.savefig(os.path.join(OUT_DIR, 'figure2c_pca.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved -> outputs/figure2c_pca.png')
    print(f'Probe direction norm: {direction.norm().item():.4f}  (should be 1.0)')
else:
    print('No probe available; skipping Figure 2c.')

In [ ]:
# Cell 8 — Save best probe to disk
if best_probe is not None:
    best_probe.save(PROBE_PATH)
    print(f'Probe saved: {PROBE_PATH}  (layer={best_probe.layer})')
    # Also write metadata
    import json
    meta = {'best_layer': int(best_layer), 'cv_accuracy': float(layer_accuracies.get(best_layer, 0)),
            'n_train': len(train_labels), 'faithful_fraction': sum(train_labels)/max(1,len(train_labels))}
    with open(os.path.join(OUT_DIR, 'probe_meta.json'), 'w') as f:
        json.dump(meta, f, indent=2)
    print('Probe metadata:', meta)
else:
    print('Warning: no probe trained; outputs/probe_best.pkl not written.')

print('Notebook 02 complete')